# Reliable LLM Systems in Production: Observability, Validation, and Failure Recovery


---

Author: Constantine (Kostyantyn) Gurnov
Org: Hyperscale.AI
Version: 0.1 Apr 28, 2026

---



## Executive Summary

Large Language Models (LLMs) are increasingly being integrated into business workflows, internal tooling, customer support systems, analytics pipelines, and decision-support environments. While these systems often demonstrate strong semantic capability, many production deployments encounter a common challenge: outputs may be useful to humans, but insufficiently reliable for machine-driven workflows.
Unlike traditional deterministic software components, LLM systems are probabilistic by nature. The same request may produce variable formatting, inconsistent structure, omitted fields, excessive verbosity, or responses that are operationally difficult to consume downstream. This creates risk when LLM outputs are embedded inside business-critical systems.

This paper presents a practical reliability framework centered on three control layers:

1. **Observability** — making model behavior visible through structured telemetry
2. **Validation** — detecting outputs that violate expected constraints
3. **Failure Recovery** — applying lightweight interventions such as retries, normalization, and controlled output shaping

A prototype reliability harness was developed to demonstrate how repeated prompts, structured logging, validation checks, and progressive controls can move an LLM pipeline from best-effort behavior toward controlled system behavior.

The goal of this paper is practical: to help engineering teams safely scale LLM-powered workflows without silent failures, brittle integrations, or hidden operational drift.


## Problem Definition

Many organizations begin LLM adoption through isolated prototypes: summarization tools, assistants, extraction pipelines, or lightweight internal automation. Early demonstrations often succeed because a human remains in the loop, manually interpreting outputs and compensating for inconsistencies.

However, once these systems are integrated into production workflows, requirements change significantly.

Production systems typically require:

* machine-readable outputs
* predictable formatting
* measurable latency
* repeatable behavior
* recoverable failures
* safe downstream integration
* auditability and traceability


This creates a gap between **semantic usefulness** and **operational reliability**.

For example:

* A human can understand a nearly-correct JSON response.
* A parser cannot.
* A human can ignore extra markdown wrappers.
* An automation pipeline may fail immediately.
* A human can detect suspicious content intuitively.
* A downstream service may ingest it silently.

As a result, many LLM initiatives stall not because the model lacks intelligence, but because the surrounding system lacks controls.

The practical engineering question becomes:

> How can probabilistic model behavior be made observable, measurable, and safe enough for production use?

---

## Why LLMs Fail in Production

The following categories represent common **possible failure modes** in early-stage LLM systems. Their exact frequency depends on provider, prompting strategy, workload design, model version, and runtime conditions. Future versions of this paper will refine prioritization using measured telemetry.


Each request can emit structured telemetry such as:

* request ID
* timestamp
* prompt version
* model identifier
* latency
* raw output
* processed output
* validation status
* detected issues
* retry count



## Observability Architecture

To improve reliability, a lightweight observability protocol was designed around the following control loop:

**observe → detect → intervene → stabilize → measure**

### Request Flow

Input Prompt → LLM Response → Validation Layer → Optional Recovery Logic → Structured Log Event → Metrics Summary

### Instrumentation Elements

Each request can emit structured telemetry such as:

* request ID
* trace ID
* timestamp
* prompt version
* model identifier
* latency
* raw output
* processed output
* validation status
* detected issues
* retry count

### Why Observability Matters

Without telemetry, teams debate anecdotes.

With telemetry, teams can answer:

* Which prompts fail most often?
* What percentage of outputs validate?
* Where does latency increase?
* Which interventions improve outcomes?
* Are failures random or patterned?


## Instrument Analysis

### Analysis Protocol
Analysis is split into Blocks. Blocks are atomic analysis that could inform our decisions. Blocks are designed sequentially, following one after another. Each block has specific goal, locked scope, expected artifacts, and exit gates that act as a data and functionality targets for the following block.

### Hypothesis

#### 1. Prompt Hardening

Clearer constraints often reduce ambiguity.

Examples:

* “Return valid JSON only”
* “Use exactly these keys”
* “No explanatory text”
  
#### 2. Output Normalization

Post-processing can remove presentation noise.

Examples:

* stripping markdown fences
* trimming prefixes
* isolating JSON candidates

#### 3. Schema Validation

Strict validation blocks malformed outputs before downstream impact.

#### 4. Controlled Retry Logic

Retries should be selective and bounded, triggered by explicit failure classes.

#### 5. Metrics Feedback Loop

Prompts, validators, and policies should evolve from measured results rather than intuition alone.

### Block 00: Configuration

In [2]:
import json
import logging
import os
import datetime as dt
import time
import uuid
from dotenv import load_dotenv

load_dotenv(".env.local")

SESSION_ID = f'{dt.datetime.now().strftime("%Y-%m-%d")}/{time.time_ns()}-{uuid.uuid4()}'

LOGS_ROOT_DIR = os.environ['LOGS_ROOT_DIR']
assert LOGS_ROOT_DIR, 'LOGS_ROOT_DIR env variable is not set. Please configure .env.local file'

LOGS_SESSION_DIR = os.path.join(LOGS_ROOT_DIR, SESSION_ID)
os.makedirs(LOGS_SESSION_DIR, exist_ok=True)


logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger("analysis")


from openai import OpenAI
client = OpenAI()

### Block 01: Baseline

#### **Block Goal** 
By the end of this block, we want a minimal observable LLM pipeline that logs:
* request ID
* input
* output
* latency
* validation status

**Locked scope**
Use:
* We use jupyter lab
* Public LLM provider API if available, otherwise placeholder first
* functions: call_llm, validate_output, block_01_baseline

**Expected Artifacts**
* one successful real model call one printed and stored structured log

#### Functions: Instrument
* call_llm
* store_log
* new_trace_id

In [59]:
import importlib
import instrument
importlib.reload(instrument)
instrument.info()

{'instrument-name': 'ai-observability',
 'instrument-version': '0.0.1',
 'updated': '2026-04-30 14:23'}

#### Functions: Analysis

In [60]:
instrument.run_experiment(
    {
  "code": "block_01_baseline",
  "prompts": [
    {
      "prompt": "Explain what AI observability is in 2 sentences.",
      "system_message": None,
      "description": "simple baseline for initial estimates",
      "runs": 1,
      "validators": []
    }
  ]
}
)

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "a797ef29-8cd2-4662-8eed-17fb3b0b904c",
  "trace_id": "1777584196395740000-a3c771bb-981b-41c0-a816-d8feeecb0571",
  "prompt": "Explain what AI observability is in 2 sentences.",
  "output": "AI observability refers to the ability to monitor, understand, and assess the performance and behavior of AI systems in real-time. It encompasses tracking metrics, diagnosing issues, and ensuring transparency in AI decision-making processes to enhance reliability and trustworthiness in AI applications.",
  "latency": 2.7988,
  "output_length": 303,
  "status": "success",
  "is_valid": true,
  "issues": []
}

SUMMARY
{
  "total_runs": 1,
  "success_count": 1,
  "valid_count": 1,
  "invalid_count": 0,
  "success_percentage": 100.0,
  "avg_latency": 2.7988
}


{'trace_id': '1777584196395740000-a3c771bb-981b-41c0-a816-d8feeecb0571',
 'runs': [{'request_id': 'a797ef29-8cd2-4662-8eed-17fb3b0b904c',
   'trace_id': '1777584196395740000-a3c771bb-981b-41c0-a816-d8feeecb0571',
   'prompt': 'Explain what AI observability is in 2 sentences.',
   'output': 'AI observability refers to the ability to monitor, understand, and assess the performance and behavior of AI systems in real-time. It encompasses tracking metrics, diagnosing issues, and ensuring transparency in AI decision-making processes to enhance reliability and trustworthiness in AI applications.',
   'latency': 2.7988,
   'output_length': 303,
   'status': 'success',
   'is_valid': True,
   'issues': []}],
 'summary': {'total_runs': 1,
  'success_count': 1,
  'valid_count': 1,
  'invalid_count': 0,
  'success_percentage': 100.0,
  'avg_latency': 2.7988}}

#### Block Summary and Exit
Exit Gate for this block:
* one successful real model call
* one printed structured log

### Block 02: Multiple Prompts + First Failure Surface

#### **Block Goal**
**Turn the single-call script into a tiny test protocol that runs several prompts and produces logs we can inspect for**:
* inconsistency
* weak answers
* formatting drift
* validation failures

**What changes in this block**

Instead of one input, we run a small batch of prompts. Use 5 prompts in our prompt set:
```
PROMPTS = [
    "Explain what AI observability is in 2 sentences.",
    "Define AI observability for a non-technical manager.",
    "List 3 risks of not monitoring LLM systems.",
    "Return a JSON object with keys: concept, risks, benefit.",
    "Summarize why tracing matters in LLM pipelines in one sentence."
]
```

These are intentionally slightly different so you can observe variation.

**This set gives us**:
* normal explanatory output
* audience shift
* list output
* structured output expectation
* brevity constraint

That is enough to expose drift.

#### Functions: Instrumentation
* call_llm (updated)

In [5]:
def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content or ""


#### Functions: Analysis

In [6]:
import time
import uuid
import json
import logging
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(message)s")

client = OpenAI()

PROMPTS = [
    "Explain what AI observability is in 2 sentences.",
    "Define AI observability for a non-technical manager.",
    "List 3 risks of not monitoring LLM systems.",
    "Return a JSON object with keys: concept, risks, benefit.",
    "Summarize why tracing matters in LLM pipelines in one sentence."
]


def validate_output(prompt: str, output: str) -> tuple[bool, list[str]]:
    issues = []

    if not output.strip():
        issues.append("empty_output")

    if len(output.strip()) < 20:
        issues.append("too_short")

    if "json object" in prompt.lower():
        try:
            json.loads(output)
        except Exception:
            issues.append("invalid_json")

    if "one sentence" in prompt.lower():
        sentence_count = output.count(".") + output.count("!") + output.count("?")
        if sentence_count > 1:
            issues.append("too_many_sentences")

    return (len(issues) == 0, issues)


def run_prompt(prompt: str, trace_id) -> dict:
    request_id = str(uuid.uuid4())
    start_time = time.time()

    try:
        response = call_llm(prompt)
        status = "success"
    except Exception as e:
        response = str(e)
        status = "error"

    latency = time.time() - start_time
    is_valid, issues = validate_output(prompt, response)

    return {
        "request_id": request_id,
        "trace_id": trace_id,
        "prompt": prompt,
        "output": response,
        "latency": round(latency, 4),
        "output_length": len(response),
        "status": status,
        "is_valid": is_valid,
        "issues": issues,
    }


def block_02_multiple_prompts() -> None:
    results = []
    trace_id = new_trace_id()
    for prompt in PROMPTS:
        result = run_prompt(prompt, trace_id)
        results.append(result)
        logging.info(json.dumps(result, indent=2))

    summary = {
        "total_runs": len(results),
        "success_count": sum(1 for r in results if r["status"] == "success"),
        "valid_count": sum(1 for r in results if r["is_valid"]),
        "invalid_count": sum(1 for r in results if not r["is_valid"]),
        "avg_latency": round(sum(r["latency"] for r in results) / len(results), 4),
    }

    block_02_log = {
        'trace_id': trace_id,
        'runs': results,
        'summary': summary,
    }
    store_log(block_02_log, 'block_02_multiple_prompts', trace_id=trace_id)

    logging.info("\nSUMMARY")
    logging.info(json.dumps(summary, indent=2))
    return block_02_log
    
result = block_02_multiple_prompts()

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "d8ea0b64-d08b-413d-9fde-81954db30fa8",
  "trace_id": "1777567568968421000-c432e758-1db6-4853-af10-4fffecfe220a",
  "prompt": "Explain what AI observability is in 2 sentences.",
  "output": "AI observability refers to the practice of monitoring and analyzing the performance, behavior, and health of AI systems throughout their lifecycle. This includes tracking metrics such as model accuracy, data quality, and system performance to ensure that AI models function as intended and to facilitate troubleshooting and continuous improvement.",
  "latency": 1.7503,
  "output_length": 347,
  "status": "success",
  "is_valid": true,
  "issues": []
}
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "7b1a065c-6de7-4f14-a6e6-1728130462c1",
  "trace_id": "1777567568968421000-c432e758-1db6-4853-af10-4fffecfe220a",
  "prompt": "Define AI observability for a n

#### Block Summary and Analysis
Read the outputs and look for:

* Did the JSON prompt actually return valid JSON? [N]
* Did the one-sentence prompt stay one sentence? [Y]
* Were any outputs too short? [N]
* Did any response style drift unexpectedly? [Y]
* Which prompt was least reliable? [JSON]

What counts as success for Block 2
We are done when we have:
* 5 logged runs
* 1 summary block
* at least 1 observed weakness or inconsistency

With iterative continuous improvements, it is enough to surface and address one weakness per cycle.

A valid issue might include:
* JSON failed once
* one-sentence output became two sentences
* list formatting varied
* tone shifted by audience

**That gives us your first failure surface.** 
Note on documentation. As you inspect the results, note 3 things:
* Most reliable prompt
* Least reliable prompt
* Most useful validation rule so far

Exit gate for the block is successful, when changes the system analysis from:
* input → output

Into:
* repeated runs → observable patterns → measurable weaknesses

### Block 03: Prompt Hardening

#### Block Goal
**Make JSON Reliable - Stabilization via Prompt Hardening and System Level Instruction**

We will choose observed weak prompt **the JSON**, and improve reliability through:
* tighter prompting
* output constraints
* simple guardrails


#### 1. Prompt hardening
The JSON request was rewritten to explicitly require:
* raw JSON only
* no markdown or commentary
* strict key definitions
* no surrounding text

In [30]:
def call_llm(prompt: str, system_message: str = None) -> str:
    messages = []
    if system_message:
        messages.append({ "role": "system", "content": system_message})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages,)
    return response.choices[0].message.content or ""

In [26]:
def validate_output_block_03_1(prompt: str, output: str) -> tuple[bool, list[str]]:
    issues = []

    if not output.strip():
        issues.append("empty_output")

    if len(output.strip()) < 20:
        issues.append("too_short")

    if "json object" in prompt.lower():
        try:
            json.loads(output)
        except Exception:
            issues.append("invalid_json")

    if "one sentence" in prompt.lower():
        sentence_count = output.count(".") + output.count("!") + output.count("?")
        if sentence_count > 1:
            issues.append("too_many_sentences")

    return (len(issues) == 0, issues)


def run_prompt(prompt: str, trace_id) -> dict:
    request_id = str(uuid.uuid4())
    start_time = time.time()

    try:
        response = call_llm(prompt)
        status = "success"
    except Exception as e:
        response = str(e)
        status = "error"

    latency = time.time() - start_time
    is_valid, issues = validate_output(prompt, response)

    return {
        "request_id": request_id,
        "trace_id": trace_id,
        "prompt": prompt,
        "output": response,
        "latency": round(latency, 4),
        "output_length": len(response),
        "status": status,
        "is_valid": is_valid,
        "issues": issues,
    }


def block_03_01_hardened_prompt(prompts) -> None:
    results = []
    trace_id = new_trace_id()
    for prompt in prompts:
        for idx in range(prompt.get('runs')):
            result = run_prompt(prompt.get('prompt'), trace_id)
            results.append(result)
            logging.info(json.dumps(result, indent=2))

    summary = {
        "total_runs": len(results),
        "success_count": sum(1 for r in results if r["status"] == "success"),
        "valid_count": sum(1 for r in results if r["is_valid"]),
        "invalid_count": sum(1 for r in results if not r["is_valid"]),
        "success_percentage": (sum(1 for r in results if r["status"] == "success") * 100 / len(results)),
        "avg_latency": round(sum(r["latency"] for r in results) / len(results), 4),
    }

    trace_log = {
        'trace_id': trace_id,
        'runs': results,
        'summary': summary,
    }
    store_log(trace_log, 'block_03_01_hardened_prompt', trace_id=trace_id)

    logging.info("\nSUMMARY")
    logging.info(json.dumps(summary, indent=2))
    return trace_log

KeyError: 'validate_output_sentences_2'

**Block 03. 1.1. Prompt Hardening: Baseline**

In [27]:
result = block_03_01_hardened_prompt(prompts = [
    {
        "prompt": "Return a JSON object with keys: concept, risks, benefit.",
        "code": "block_03_01",
        "description": "I. Baseline",
        "runs": 1
    }
])

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "a874bbb1-11a8-4048-91da-0ac0d8fb1cd6",
  "trace_id": "1777577849346273000-2acfdc45-af43-4508-9da7-c8345334281e",
  "prompt": "Return a JSON object with keys: concept, risks, benefit.",
  "output": "Here is a JSON object with the specified keys:\n\n```json\n{\n  \"concept\": \"Artificial Intelligence in Healthcare\",\n  \"risks\": [\n    \"Data privacy concerns\",\n    \"Bias in algorithms\",\n    \"Job displacement\",\n    \"Over-reliance on technology\",\n    \"Lack of transparency in decision-making\"\n  ],\n  \"benefit\": [\n    \"Improved diagnostic accuracy\",\n    \"Personalized treatment plans\",\n    \"Increased efficiency in healthcare delivery\",\n    \"Enhanced patient monitoring\",\n    \"Reduced costs through predictive analytics\"\n  ]\n}\n```",
  "latency": 2.8643,
  "output_length": 521,
  "status": "success",
  "is_valid": false,
  "issues": [
    "invalid_json"
  ]
}

SU

**Block 03. 1.2. Prompt Hardened**
* raw JSON only
* no markdown or commentary
* strict key definitions

```text
Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.
```

In [28]:
result = block_03_01_hardened_prompt(prompts = [
    {
        "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings",
        "code": "block_03_01",
        "description": "Prompt Hardened",
        "runs": 1
    }
])

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "b7913858-7643-454c-8a08-bfe37c7a0939",
  "trace_id": "1777577855832532000-eb486dc3-823b-4b70-8515-f13874757d08",
  "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings",
  "output": "{\n  \"concept\": \"Sustainable Agriculture\",\n  \"risks\": [\n    \"Higher upfront costs for eco-friendly materials\",\n    \"Potential yield losses during the transition period\",\n    \"Dependence on market demand for organic products\",\n    \"Vulnerability to climate change impacts\"\n  ],\n  \"benefit\": [\n    \"Improved soil health and biodiversity\",\n    \"Reduced greenhouse gas emissions\",\n    \"Increased resilience to pests and diseases\",\n    \"Enhanced food security and nutrition\"\n  ]\n}",
  "latency": 2.5671,
  "output_length": 474,
  "status": "success

**Block 03. 1.2. Prompt Hardened with Re-runs**

* Add multiple re-runs to verify stability of the improvement

In [29]:
result = block_03_01_hardened_prompt(prompts = [
    {
        "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings",
        "code": "block_03_01",
        "description": "Prompt Hardened",
        "runs": 3
    }
])

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
{
  "request_id": "4fe8159e-2e02-48da-9d0d-c4d1395e0e48",
  "trace_id": "1777577872095248000-a268e343-f89f-4093-bb84-ac8c9ee34016",
  "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings",
  "output": "```json\n{\n  \"concept\": \"Renewable Energy Sources\",\n  \"risks\": [\n    \"Intermittency of energy supply\",\n    \"High initial investment costs\",\n    \"Land use and environmental impact\",\n    \"Technological reliance and complexity\"\n  ],\n  \"benefit\": [\n    \"Reduction in greenhouse gas emissions\",\n    \"Sustainable and inexhaustible energy supply\",\n    \"Job creation in the renewable energy sector\",\n    \"Decreased dependence on fossil fuels\"\n  ]\n}\n```",
  "latency": 2.8847,
  "output_length": 443,
  "status": "success",
  "is_valid": true,
  "iss

#### 2. System-level instruction
Better Approach — Separate System + User Messages
* A system message was added to enforce JSON-only behavior and reduce conversational responses.

That gives us the before/after observation for Prompt Hardening and System-level instruction

In [31]:
def call_llm(prompt: str, system_message: str = None) -> str:
    messages = []
    if system_message:
        messages.append({ "role": "system", "content": system_message})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages,)
    return response.choices[0].message.content or ""

In [32]:
def validate_output_block_03_1(prompt: str, output: str) -> tuple[bool, list[str]]:
    issues = []

    if not output.strip():
        issues.append("empty_output")

    if len(output.strip()) < 20:
        issues.append("too_short")

    if "json object" in prompt.lower():
        try:
            json.loads(output)
        except Exception:
            issues.append("invalid_json")

    if "one sentence" in prompt.lower():
        sentence_count = output.count(".") + output.count("!") + output.count("?")
        if sentence_count > 1:
            issues.append("too_many_sentences")

    return (len(issues) == 0, issues)


def run_prompt(prompt: dict, trace_id) -> dict:
    request_id = str(uuid.uuid4())
    start_time = time.time()

    try:
        response = call_llm(prompt=prompt.get('prompt'), system_message=prompt.get('system_message'))
        status = "success"
    except Exception as e:
        response = str(e)
        status = "error"

    latency = time.time() - start_time
    is_valid, issues = validate_output(prompt, response)

    return {
        "request_id": request_id,
        "trace_id": trace_id,
        "prompt": prompt,
        "output": response,
        "latency": round(latency, 4),
        "output_length": len(response),
        "status": status,
        "is_valid": is_valid,
        "issues": issues,
    }


def block_03_02_hardened_prompt(prompts) -> None:
    results = []
    trace_id = new_trace_id()
    for prompt in prompts:
        for idx in range(prompt.get('runs')):
            result = run_prompt(prompt, trace_id)
            results.append(result)
            logging.info(json.dumps(result, indent=2))

    summary = {
        "total_runs": len(results),
        "success_count": sum(1 for r in results if r["status"] == "success"),
        "valid_count": sum(1 for r in results if r["is_valid"]),
        "invalid_count": sum(1 for r in results if not r["is_valid"]),
        "success_percentage": (sum(1 for r in results if r["status"] == "success") * 100 / len(results)),
        "avg_latency": round(sum(r["latency"] for r in results) / len(results), 4),
    }

    trace_log = {
        'trace_id': trace_id,
        'runs': results,
        'summary': summary,
    }
    store_log(trace_log, 'block_03_02_hardened_prompt', trace_id=trace_id)

    logging.info("\nSUMMARY")
    logging.info(json.dumps(summary, indent=2))
    return trace_log

In [ ]:
SYSTEM_MESSAGE = (
        "You are a precise API that returns only raw JSON when asked for JSON. "
        "Do not include markdown, code fences, commentary, or any extra text."
    )

In [ ]:
import datetime as dt
import os
import time
import uuid
import json
import logging

from openai import OpenAI

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(message)s")

load_dotenv(".env.local")


client = OpenAI()

def validate_output(prompt: str, output: str) -> tuple[bool, list[str]]:
    issues = []

    if not output.strip():
        issues.append("empty_output")

    if len(output.strip()) < 20:
        issues.append("too_short")

    # VI. Update Validation Logic
    if "json" in prompt.lower():

        try:
            candidate = extract_json_candidate(output)
            parsed = json.loads(candidate)

            # VII. Optional but Strong — Log Raw Output vs Extracted JSON
            json_candidate = extract_json_candidate(output) if "json" in prompt.lower() else None
            # TODO: Log it


            required_keys = {"concept", "risks", "benefit"}
            if set(parsed.keys()) != required_keys:
                issues.append("wrong_keys")

            if not isinstance(parsed.get("concept"), str):
                issues.append("concept_not_string")

            if not isinstance(parsed.get("risks"), list) or not all(isinstance(x, str) for x in parsed["risks"]):
                issues.append("risks_not_string_array")

            if not isinstance(parsed.get("benefit"), list) or not all(isinstance(x, str) for x in parsed["benefit"]):
                issues.append("benefit_not_string_array")

        except Exception:
            issues.append("invalid_json")

    if "one sentence" in prompt.lower():
        sentence_count = output.count(".") + output.count("!") + output.count("?")
        if sentence_count > 1:
            issues.append("too_many_sentences")

    return (len(issues) == 0, issues)



if __name__ == "__main__":
    main()


In [ ]:
import datetime as dt
import os
import time
import uuid
import json
import logging

from openai import OpenAI

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(message)s")

load_dotenv(".env.local")

CASE_STUDY_BLOCK = f"block3-{time.time_ns()}"


PROMPTS = [
    # "Return a JSON object with keys: concept, risks, benefit.",

    # II. First Prompt Improvement
    # "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings."

    # IV. Stronger JSON Prompt
    # "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings."

    # VIII. Recommended Test Prompt Set for This Block
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings."
]

client = OpenAI()


# III. Better Approach — Separate System + User Messages
# def call_llm(prompt: str) -> str:
#     response = client.chat.completions.create(model="gpt-4o-mini",messages=[{"role": "user", "content": prompt}],)
#     return response.choices[0].message.content or ""
def call_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{ "role": "system", "content": CASE_STUDY_SYSTEM_MESSAGE,} , {"role": "user", "content": prompt}, ],
    )
    return response.choices[0].message.content or ""

# V. Add a Cleanup Layer (Very Useful for Case Study)
def extract_json_candidate(output: str) -> str:
    output = output.strip()

    if output.startswith("```json"):
        output = output.removeprefix("```json").strip()
    if output.startswith("```"):
        output = output.removeprefix("```").strip()
    if output.endswith("```"):
        output = output[:-3].strip()

    start = output.find("{")
    end = output.rfind("}")

    if start != -1 and end != -1 and end > start:
        return output[start:end + 1]

    return output




def run_prompt(prompt: str) -> dict:
    request_id = str(uuid.uuid4())
    start_time = time.time()

    try:
        response = call_llm(prompt)
        response = extract_json_candidate(response)
        status = "success"
    except Exception as e:
        response = str(e)
        status = "error"

    latency = time.time() - start_time
    is_valid, issues = validate_output(prompt, response)

    log = {
        "request_id": request_id,
        "prompt": prompt,
        "output": response,
        "latency": round(latency, 4),
        "output_length": len(response),
        "status": status,
        "is_valid": is_valid,
        "issues": issues,
    }
    persist_log(log)
    return log


def main() -> None:
    results = []

    for prompt in PROMPTS:
        result = run_prompt(prompt)
        results.append(result)
        logging.info(json.dumps(result, indent=2))

    summary = {
        "total_runs": len(results),
        "system_note": CASE_STUDY_NOTE,
        "system_message": CASE_STUDY_SYSTEM_MESSAGE,
        "success_count": sum(1 for r in results if r["status"] == "success"),
        "valid_count": sum(1 for r in results if r["is_valid"]),
        "invalid_count": sum(1 for r in results if not r["is_valid"]),
        "avg_latency": round(sum(r["latency"] for r in results) / len(results), 4),
    }

    persist_log(log=summary, fprefix="summary")

    logging.info("\nSUMMARY")
    logging.info(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
PROMPTS =[
]
    {
        "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
        "description": "IV. Stronger JSON Prompt"
        "runs": 1,
    },
    {
        "prompt": "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
        "description": "IV. Stronger JSON Prompt"
        "runs": 3,
    },
    

    # 
    # 

    # IV. Stronger JSON Prompt
    # 
    # VIII. Recommended Test Prompt Set for This Block
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings.",
    "Return only valid JSON with exactly these top-level keys: concept, risks, benefit. concept must be a string. risks must be an array of strings. benefit must be an array of strings."
]

### Block 04: Schema Validation and Recovery

#### 1. Output normalization layer  
   A lightweight post-processing function was introduced to:
   - extract JSON from wrapped responses
   - remove markdown fences
   - isolate valid JSON substrings

In [11]:
# V. Add a Normalization (Cleanup Layer)
def extract_json_candidate(output: str) -> str:
    output = output.strip()

    if output.startswith("```json"):
        output = output.removeprefix("```json").strip()
    if output.startswith("```"):
        output = output.removeprefix("```").strip()
    if output.endswith("```"):
        output = output[:-3].strip()

    start = output.find("{")
    end = output.rfind("}")

    if start != -1 and end != -1 and end > start:
        return output[start:end + 1]

    return output

#### 2. Schema validation  
   Outputs were validated against expected structure:
   - required keys: concept, risks, benefit
   - correct data types (string, array of strings)

In [ ]:
def validate_output(prompt: str, output: str) -> tuple[bool, list[str]]:
    issues = []

    if not output.strip():
        issues.append("empty_output")

    if len(output.strip()) < 20:
        issues.append("too_short")

    # VI. Update Validation Logic
    if "json" in prompt.lower():

        try:
            candidate = extract_json_candidate(output)
            parsed = json.loads(candidate)

            # VII. Optional but Strong — Log Raw Output vs Extracted JSON
            json_candidate = extract_json_candidate(output) if "json" in prompt.lower() else None
            # TODO: Log it


            required_keys = {"concept", "risks", "benefit"}
            if set(parsed.keys()) != required_keys:
                issues.append("wrong_keys")

            if not isinstance(parsed.get("concept"), str):
                issues.append("concept_not_string")

            if not isinstance(parsed.get("risks"), list) or not all(isinstance(x, str) for x in parsed["risks"]):
                issues.append("risks_not_string_array")

            if not isinstance(parsed.get("benefit"), list) or not all(isinstance(x, str) for x in parsed["benefit"]):
                issues.append("benefit_not_string_array")

        except Exception:
            issues.append("invalid_json")

    if "one sentence" in prompt.lower():
        sentence_count = output.count(".") + output.count("!") + output.count("?")
        if sentence_count > 1:
            issues.append("too_many_sentences")

    return (len(issues) == 0, issues)

## Analysis Results

Improving Reliability of LLM Outputs via Observability and Structured Validation

### 1. Problems Observed
* Large Language Models often produce outputs that are semantically correct but not reliably structured for downstream systems.
* In this case, the goal was to evaluate how consistently an LLM can return machine-readable JSON under repeated runs.
* The initial assumption was that simple prompting would be sufficient, but early results showed variability in formatting and output structure.
* This creates a risk for systems that depend on strict output formats (e.g., JSON parsing).

### 2. Baseline System (What was built first)
A minimal LLM pipeline was implemented:

Input → LLM → Output

Each request was instrumented with:
- request ID
- latency measurement
- raw output logging

A small test protocol and instrument executed multiple prompts, including a JSON-generation task.
Logs were persisted as structured JSON events to enable comparison across runs.

### 3. Observed Failures (This is key)
During repeated runs, the following issues were observed:

1. JSON responses were wrapped in natural language explanations
2. Markdown code fences (```json) were included
3. Additional commentary appeared before and after the JSON object
4. Output format varied across identical prompts

Although the model understood the task, the outputs were not consistently machine-readable.

Example output (truncated):
```
"output": "Sure! Here is a JSON object with the specified keys:\n\n```json\n [{ ... ]\n}\n```\n\n ... for your specific context!",
```

Here's a JSON object structure with the requested keys:

```json
{
  "request_id": "4e33c5a2-fb27-4779-9f47-efd87162f302",
  "trace_id": "1777567568968421000-c432e758-1db6-4853-af10-4fffecfe220a",
  "prompt": "Return a JSON object with keys: concept, risks, benefit.",
  "output": "Sure! Here is a JSON object with the specified keys:\n\n```json\n [{... ]\n}\n```\n\n ... for your specific context!",
  "latency": 2.6998,
  "output_length": 726,
  "status": "success",
  "is_valid": false,
  "issues": [
    "invalid_json"
  ]
}
```

### 4. Intervention (What changed)
To improve reliability, several interventions were introduced:

1. Prompt tightening  
   The JSON request was rewritten to explicitly require:
   - raw JSON only
   - no markdown or commentary
   - strict key definitions

2. System-level instruction  
   A system message was added to enforce JSON-only behavior and reduce conversational responses.

3. Output normalization layer  
   A lightweight post-processing function was introduced to:
   - extract JSON from wrapped responses
   - remove markdown fences
   - isolate valid JSON substrings

4. Schema validation  
   Outputs were validated against expected structure:
   - required keys: concept, risks, benefit
   - correct data types (string, array of strings)

### 5. Results (Before / After)
After these changes:

- JSON validity improved across repeated runs
- malformed outputs became detectable and recoverable
- output consistency increased

The system moved from non-deterministic “best-effort” loosely formatted outputs, to a more controlled, measurable, verifiable output behavior.

Even when the model deviated, the normalization and validation layers ensured that only valid, machine-readable data was passed downstream.

This pattern is critical for production systems where LLM outputs are consumed programmatically rather than read by humans.


| Scenario              | Success Rate | Avg Latency |
| --------------------- | -----------: | ----------: |
| Baseline              |     75%      |    1.6583 |
| Prompt Hardened       |     [INSERT] |    3.2047 |
| Validation + Recovery |     [INSERT] |    [INSERT] |

### 6. Key Takeaways
- LLM outputs should not be treated as guaranteed structured APIs
- Observability is required to detect variability and failure modes
- Prompting alone is insufficient for reliability
- Validation and normalization layers are essential for production use

### 7. Case Study Summary
The final system consisted of:

- a lightweight evaluation instrument (CLI-based)
- prompt and system message separation
- structured logging (JSONL)
- validation and normalization layer


## Suggested Production Rollout


### Phase 1 — Human-Assisted Prototype

* manual review
* exploratory prompts
* no critical dependencies

### Phase 2 — Instrumented Pilot

* structured logs
* latency tracking
* validation rules
* limited workflow integration

### Phase 3 — Controlled Production

* schema contracts
* alerting
* bounded retries
* rollback procedures
* ownership model

### Phase 4 — Continuous Optimization

* prompt versioning
* telemetry-driven tuning
* provider comparison
* cost/performance balancing

### Rollout Checklist
Successfull integration for Non-deterministic System Should demonstrates:

1. Observability
* request-level tracing
* structured logs
* latency + validation

2. Failure detection
* invalid JSON
* format drift
* constraint violations

3. Stabilization
* prompt tightening
* system-level constraints
* post-processing layer

4. Measurement
* per-run validation
* aggregate summary

A complete production grade loop:
* observe → detect → intervene → measure

## Appendix: Logs and Data Artifacts Structure

Logs and Generated Data Artifacts have **3-groups split**


### Group 1: In the notebooks, and case study documents
Included:
* a few short excerpts
* summary metrics
* before/after examples
* Placement in Documentation, Whitepaper, Articles, Case study Writeup
* As part of documentation analysis, in source code

### Group 2: Representative logs in a sanitized samples
Included:
* schemas
* prompt definitions
* schemas
* validation logic
* **representative sanitized logs and samples**

```text
repo/
  app.py
  README.md
  docs/
  prompts/
  logs/
    samples/
      baseline_sample.jsonl
      block_01_stabilized_sample.jsonl
      ...
```

Select as artifacts:
* Provide **clarity**
* Small, readable, intentionally chosen
* Safe to publish
* Enough to demonstrate the evidence


### Group 3. Full/raw run data outside the main repo, as a data archive
* **audit trail** if required
* full run logs
* repeated test batches
* experimental outputs
* larger datasets
* batch outputs
* experiment history
* Separate private repo
* Preserved for release asset bundle
* Available by requests

Structural template (separate repo):

```text
case-studies-data-archive/ai-observability
  [session-id]/
      [block]/
      summaries/
```